# Quantum Sensing versus Quantum simulation
The error of simulation is
$$
    \mathcal E_{sim}(\boldsymbol \delta):=\frac{1}{2}\left\|\rho_{sim}-\rho_{ideal}\right\|_1.
$$
For a small error vector $\boldsymbol{\delta}$, the leading-order simulation error is determined by the projection of the error onto the QFIM metric tensor:
$$
\mathcal{E}_{\text{sim}}(\boldsymbol{\delta}) \approx \frac{1}{2} \sqrt{ \boldsymbol{\delta}^\top \mathcal{F}_Q(\rho) \boldsymbol{\delta} }.
$$
For a pure parameterized state $\ket{\psi}=\ket{\psi(\vec x)}$, the entries of the QFIM is 
$$
    \mathcal F_{ab}=4\mathrm{Re}(\braket{\partial_a \psi|\partial_b\psi}-\braket{\partial_a \psi|\psi}\braket{\psi|\partial_b\psi}),
$$
and the trace distance is 
$$
    \mathcal E_{sim}(\delta)=\sqrt{1-\left|\braket{\psi_{ideal}|\psi_{real}} \right|^2}
$$
## Single variable
We first consider the Hamiltonian with form
$$
H=\theta H_0.
$$
Given the initial state $\ket{\psi_0}$, the QIF of $\ket{\psi_t(\theta)}$ is:
$$
    \mathcal F(\theta, \psi_0)=4t^2(\bra{\psi_t(\theta)}H_0^2\ket{\psi_t(\theta)}-\bra{\psi_t(\theta)}H_0\ket{\psi_t(\theta)}^2)=t^2\mathrm{Var}(E).
$$
Include a small error $\delta$, then the simulation error
$$
\mathcal E_{sim}(\delta)=\sqrt{1-\left|\bra{\psi_0}e^{-i\delta H_0 t}\ket{\psi_0}\right|^2}.
$$

In [1]:
import numpy as np
from scipy.linalg import expm
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.linalg import expm
from src import Nearest_Neighbour_1d, expH
import matplotlib.pyplot as plt

def QFI_single_variable(h0,psi,theta,time):
    Evolution_operator=expH(h0.ham,time*theta)
    psi_t=psi.evolve(Evolution_operator).data
    H0=h0.ham.to_matrix()
    H0_squared=H0@H0
    qfi=4*time**2*(psi_t.conj().T@H0_squared@psi_t- (psi_t.conj().T@H0@psi_t)**2).real
    return qfi

def Simulation_error_single_variable(delta, h0, psi, time):
    Evolution_operator=expH(h0.ham,time*delta)
    psi_t=psi.evolve(Evolution_operator).data
    return np.sqrt(1-abs(psi.data.conj().T@psi_t)**2)

### 1D QIMF:
We first consider the 1D Ising Model
$$
    H_0=-J\sum_{j=1}^{N-1}Z_jZ_{j+1}-h_x\sum_{j=1}^N X_j.
$$

In [2]:
#==========Setting: H_0==========#
N=2
Jz=1.0
hx=0.5
h_0=Nearest_Neighbour_1d(n=N,Jz=Jz,hx=hx)
t=1
#==========Setting: Initial state and parameters==========#
psi=Statevector.from_label('0'*N)
delta_list=np.linspace(0, 1, 50)
theta=1
qfis=[]
errors=[]
for delta in delta_list:
    qfi=.5*delta*np.sqrt(QFI_single_variable(h_0,psi,theta=theta,time=t))
    error=Simulation_error_single_variable(delta,h_0,psi,time=t)
    qfis.append(qfi)
    errors.append(error)
print(qfis)
print(errors)
#plt.plot(delta_list, qfis, label='QFI')

[np.float64(0.0), np.float64(0.014430750636460156), np.float64(0.028861501272920313), np.float64(0.04329225190938047), np.float64(0.057723002545840625), np.float64(0.07215375318230077), np.float64(0.08658450381876094), np.float64(0.1010152544552211), np.float64(0.11544600509168125), np.float64(0.1298767557281414), np.float64(0.14430750636460155), np.float64(0.1587382570010617), np.float64(0.17316900763752188), np.float64(0.187599758273982), np.float64(0.2020305089104422), np.float64(0.21646125954690232), np.float64(0.2308920101833625), np.float64(0.24532276081982266), np.float64(0.2597535114562828), np.float64(0.274184262092743), np.float64(0.2886150127292031), np.float64(0.3030457633656633), np.float64(0.3174765140021234), np.float64(0.3319072646385836), np.float64(0.34633801527504376), np.float64(0.3607687659115039), np.float64(0.375199516547964), np.float64(0.3896302671844242), np.float64(0.4040610178208844), np.float64(0.41849176845734454), np.float64(0.43292251909380464), np.float

## Multi-paras. QFIM
We use GPU to calculate the matrix. First, we calculate $\ket{\psi(\delta)}$ and $\ket{\partial_a \psi(\delta)}$

In [3]:
import jax
import jax.numpy as jnp
from jax.scipy.linalg import expm
psi=Statevector.from_label('0'*N).data
time = 1
H_list=jnp.stack([jnp.array([[0,0,0,0],[0,0,0,0],[0,0,0,0],[0,0,0,0]]),h_0.ham.to_matrix()])
deltas=jnp.array([0.2])
def construct_H(delta,H_list):
    return H_list[0]+jnp.einsum("k,kij->ij",delta, H_list[1:])

@jax.jit
def state(delta, psi0, H_list,time):
    H = construct_H(delta, H_list)
    return expm(-1j*time*H)@psi0

@jax.jit
def derivative(delta, psi0, H_list,time):
    y = state(delta,psi0,H_list,time)
    dy = jax.jacfwd(lambda d: state(d,psi0,H_list,time))(delta)
    return y,dy
# Here y, dy are respect to psi(delta) and partial derivativation.

@jax.jit
def core(y,dy):
    A = dy.conj().T @ dy
    B = dy.conj().T @ y
    return 4*(A - jnp.outer(B,B.conj())).real

def General_QFI(delta,psi0,H_list,time):
    y,dy = derivative(delta, psi0, H_list,time)
    return core(y,dy)

def General_Simulation_error(delta, psi0, H_list,time):
    Real = state(delta,psi0,H_list,time)
    ideal = state(jnp.zeros_like(delta),psi0,H_list,time)
    return jnp.sqrt(1-(ideal.conj().T@Real).real**2)

print(General_QFI(deltas,psi,H_list,time))
print(General_Simulation_error(deltas,psi,H_list,time))



E0605 12:27:11.649435    1295 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0605 12:27:11.668548    1076 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


[[1.999999]]
0.24244012


E0501 12:14:18.575559    1992 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0501 12:14:18.585582    1517 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


(-4.7683716e-07+0j)



For any quantum state $\rho$, any error vector $\boldsymbol{\delta}$, and any unbiased estimation protocol using $M$ shots, the product of the simulation error and the estimation precision along the error direction is bounded from below:
$$
    \mathcal E_{sim}(\boldsymbol \delta)^2\cdot V_{eff}\ge\frac{1}{4M}
$$
where $V_{\text{eff}} := (\boldsymbol{\delta}^\top \Sigma^{-1} \boldsymbol{\delta})^{-1}$ represents the generalized variance of the estimator projected along the error direction. Consider an unbiased estimator $\hat{\boldsymbol{\delta}}$ constructed from $M$ independent measurements, $\Sigma = \text{Cov}(\hat{\boldsymbol{\delta}})$ is the covariance matrix.
